**By Rupesh Panwar ([VerBodh](https://verbodh.com))** · [LinkedIn](https://www.linkedin.com/in/rupesh-panwar/)

VerBodh is an AI and LLM apps startup, seeded in Singapore.

**How this build differs from the course's default route**

1. **Amazon Cognito for identity and tokens.** Handlers sign in to the dashboard through a Cognito user pool, and the
   app checks Cognito's tokens, in place of Supabase Auth.
2. **Amazon RDS for PostgreSQL in place of Supabase.** The claims data stays inside the customer's AWS account and
   never leaves the AWS ecosystem.
3. **No IAM users anywhere.** The customer doesn't allow long-lived IAM user keys, so every human call in this
   solution, this notebook included, signs in through AWS IAM Identity Center, with SSO and MFA. The AWS services
   themselves use IAM roles.

# Week 3 Day 4 challenge: the agent on Runtime against the agent on the laptop

The challenge from the end of `labs/week3/week3day4_lab2.ipynb`: run an evaluation of the held-out set with the agent
on AgentCore Runtime, compare its time per claim and its cost with the same evaluation on the laptop, then look at
the session in the runtime's logs.

**What this notebook does**

1. Signs in to the deployed app as Dana through Cognito, as the deployed dashboard does, and starts an evaluation of
   the held-out set's five tiny claims. The app on Lambda sends the work to the agent on Runtime.
2. Starts the backend, the agent and the context layer on the laptop, on the direct route with memory off, signs in
   to local Supabase Auth, and runs the same evaluation there.
3. Compares the two runs' measures from the app's own records: seconds and cost per claim, the whole run's time, and
   accuracy.
4. Reads the runtime's log group in CloudWatch for the time of the Runtime evaluation, and shows its sessions.

**Before you run it:** finish Parts 1 to 4 of the Day 4 Lab 2, and don't run its Part 6 teardown yet. Nothing else
may be running on the deployed app, since it runs one dispatch or evaluation at a time. Stop `npm run dev` too,
since this notebook starts its own backend on port 8000. It uses the five tiny claims, not all 25, and costs about
two cents: two five-claim evaluations with the judge.

## Setup

In [ ]:
import truststore

# Trust the certificates in the Windows certificate store, so HTTPS works behind a network that inspects traffic.
truststore.inject_into_ssl()

import json
import os
import shutil
import ssl
import subprocess
import sys
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path

import boto3
import httpx
from dotenv import load_dotenv

REPO = Path("../..").resolve()
load_dotenv(REPO / ".env", override=True)

session = boto3.Session()
print(f"AWS account {session.client('sts').get_caller_identity()['Account']} in {session.region_name}")

### The Windows certificates as a file

The servers this notebook starts are programs of their own, which can't use `truststore`. Behind Cloudflare WARP
they need the Windows certificates as a file. On a Mac, or without WARP, this does no harm.

In [ ]:
CERTS = Path(tempfile.gettempdir()) / "undercover_windows_certs.pem"
if sys.platform == "win32":
    pems = [ssl.DER_cert_to_PEM_cert(der) for store in ("ROOT", "CA") for der, kind, _ in ssl.enum_certificates(store) if kind == "x509_asn"]
    CERTS.write_text("".join(pems), encoding="ascii")
    for name in ("SSL_CERT_FILE", "REQUESTS_CA_BUNDLE", "AWS_CA_BUNDLE", "NODE_EXTRA_CA_CERTS"):
        os.environ[name] = str(CERTS)
    print(f"Wrote the Windows certificates to {CERTS}")

## Shared steps

`TINY = True` keeps both evaluations to the held-out set's five tiny claims. Both use the agent's default model,
since the runtime doesn't have `MODEL_NAME`.

In [ ]:
TINY = True
AGENTS = 5
START = {"n": AGENTS, "eval_set": "held_out", "tiny": TINY}
MEASURES = ["status", "model", "accuracy", "decision_accuracy", "straight_through", "seconds_per_claim", "cost_per_claim", "judge_cost", "run_seconds"]


def wait_for_evaluation(client, run_id, minutes=15):
    """The evaluation's measures once it's no longer running."""
    for _ in range(minutes * 6):
        evaluation = client.get(f"/api/evaluations/{run_id}").json()
        if evaluation["status"] != "running":
            return {k: evaluation[k] for k in MEASURES}
        time.sleep(10)
    raise TimeoutError(f"Evaluation {run_id} is still running after {minutes} minutes")

## Part 1: the evaluation with the agent on Runtime

Dana signs in through the user pool with `USER_PASSWORD_AUTH`, as the deployed dashboard does, and the app checks
the ID token. The password comes from `HANDLER_PASSWORD` in `.env` and is never printed.

In [ ]:
lambda_client = session.client("lambda")
app_url = lambda_client.get_function_url_config(FunctionName="undercover-app")["FunctionUrl"].rstrip("/")
settings = lambda_client.get_function_configuration(FunctionName="undercover-app")["Environment"]["Variables"]
runtime_arn = settings.get("UNDERCOVER_AGENT_RUNTIME")
assert runtime_arn, "The app on Lambda isn't pointed at the runtime: run Part 4 of the lab"

cognito = session.client("cognito-idp")
signed_in = cognito.initiate_auth(
    ClientId=os.environ["AWS_COGNITO_CLIENT_ID"],
    AuthFlow="USER_PASSWORD_AUTH",
    AuthParameters={"USERNAME": os.environ["HANDLER_EMAIL"], "PASSWORD": os.environ["HANDLER_PASSWORD"]},
)
deployed = httpx.Client(base_url=app_url, headers={"Authorization": f"Bearer {signed_in['AuthenticationResult']['IdToken']}"}, timeout=60)
print("Signed in as", deployed.get("/api/handlers/me").json()["name"], "on", app_url)
print("The app sends the agent's work to", runtime_arn.split("/")[-1])

In [ ]:
runtime_started = datetime.now(timezone.utc)
reply = deployed.post("/api/evaluations", json=START)
assert reply.status_code == 200, reply.text
runtime_run = reply.json()["run_id"]
print(f"Started evaluation {runtime_run} on the deployed app")

on_runtime = wait_for_evaluation(deployed, runtime_run)
runtime_finished = datetime.now(timezone.utc)
on_runtime

## Part 2: the same evaluation with the agent on the laptop

The notebook starts the three parts `npm run dev` would, except the dashboard: the backend on port 8000, the agent's
AgentCore app on port 8080 and the context layer on 8006. It blanks the Gateway, the Memory and the runtime's ARN
for them, so the agent takes the direct route to the context layer and to the app on the laptop, with memory off.
The database is the laptop's own.

In [ ]:
local_env = os.environ | {"UNDERCOVER_GATEWAY_URL": "", "UNDERCOVER_MEMORY_ID": "", "UNDERCOVER_AGENT_RUNTIME": "", "UNDERCOVER_COPILOT_RUNTIME": ""}
local_env.pop("MODEL_NAME", None)
local_env.pop("VIRTUAL_ENV", None)
uv = shutil.which("uv")
commands = {
    "backend": [uv, "run", "-m", "uvicorn", "backend.main:app", "--port", "8000"],
    "agent": [uv, "run", "-m", "uvicorn", "agent.entrypoint:app", "--host", "127.0.0.1", "--port", "8080"],
    "context": [uv, "run", "-m", "context"],
}
logs = {name: open(Path(tempfile.gettempdir()) / f"undercover_{name}.log", "w") for name in commands}
servers = {name: subprocess.Popen(command, cwd=REPO, env=local_env, stdout=logs[name], stderr=subprocess.STDOUT) for name, command in commands.items()}

for name, url in {"backend": "http://127.0.0.1:8000/api/health", "agent": "http://127.0.0.1:8080/ping"}.items():
    for _ in range(60):
        try:
            if httpx.get(url, timeout=2).is_success:
                print(f"The {name} is up")
                break
        except httpx.HTTPError:
            time.sleep(2)
    else:
        raise RuntimeError(f"The {name} didn't start: see {logs[name].name}")

In [ ]:
token = httpx.post(
    f"{os.environ['VITE_SUPABASE_URL']}/auth/v1/token?grant_type=password",
    headers={"apikey": os.environ["VITE_SUPABASE_PUBLISHABLE_KEY"]},
    json={"email": os.environ["HANDLER_EMAIL"], "password": os.environ["HANDLER_PASSWORD"]},
    timeout=20,
).json()["access_token"]
laptop = httpx.Client(base_url="http://127.0.0.1:8000", headers={"Authorization": f"Bearer {token}"}, timeout=60)
print("Signed in as", laptop.get("/api/handlers/me").json()["name"], "on the laptop")

reply = laptop.post("/api/evaluations", json=START)
assert reply.status_code == 200, reply.text
laptop_run = reply.json()["run_id"]
print(f"Started evaluation {laptop_run} on the laptop")

on_laptop = wait_for_evaluation(laptop, laptop_run)
on_laptop

In [ ]:
# uv starts each server as a child process, so stop the whole tree: on Windows, terminate() stops only uv.
for server in servers.values():
    if sys.platform == "win32":
        subprocess.run(["taskkill", "/T", "/F", "/PID", str(server.pid)], capture_output=True, check=False)
    else:
        server.terminate()
for server in servers.values():
    server.wait(timeout=30)
for log in logs.values():
    log.close()
print("Stopped the backend, the agent and the context layer")

## Part 3: the two side by side

In [ ]:
rows = ["model", "status", "accuracy", "decision_accuracy", "straight_through", "seconds_per_claim", "run_seconds", "cost_per_claim", "judge_cost"]


def shown(value):
    return f"{value:.4f}" if isinstance(value, float) else str(value)


print(f"{'':20} {'Runtime':>18} {'Laptop':>18}")
for row in rows:
    print(f"{row:20} {shown(on_runtime[row]):>18} {shown(on_laptop[row]):>18}")

## Part 4: the session in the runtime's logs

Each runtime writes to its own log group, `/aws/bedrock-agentcore/runtimes/<runtime id>-DEFAULT`. The app starts a
new Runtime session for each piece of work it sends, so the evaluation is one session that starts the run, plus the
pings Runtime sends while the work goes on. This lists the log streams written during the Runtime evaluation, and
the lines that name its claims.

In [ ]:
logs_client = session.client("logs")
runtime_id = runtime_arn.split("/")[-1]
group = f"/aws/bedrock-agentcore/runtimes/{runtime_id}-DEFAULT"
start_ms = int(runtime_started.timestamp() * 1000) - 60_000
end_ms = int(runtime_finished.timestamp() * 1000) + 60_000

events, token = [], None
while True:
    page = logs_client.filter_log_events(logGroupName=group, startTime=start_ms, endTime=end_ms, **({"nextToken": token} if token else {}))
    events += page["events"]
    token = page.get("nextToken")
    if not token:
        break

streams = sorted({e["logStreamName"] for e in events})
print(f"{len(events)} log lines in {len(streams)} streams of {group}")
for stream in streams:
    print("  ", stream)

In [ ]:
mentions = [e for e in events if f"E{runtime_run}-" in e["message"] or "session" in e["message"].lower()]
for event in mentions[:25]:
    when = datetime.fromtimestamp(event["timestamp"] / 1000, timezone.utc).strftime("%H:%M:%S")
    print(when, event["message"].strip()[:200])
print(f"... {len(mentions)} lines name the evaluation's claims or a session")

## What I found

*From the run on 4 November 2026: the held-out set's five tiny claims, five claim agents, `openai/gpt-6-luna` on both.*

| | Runtime | Laptop |
|---|---|---|
| Seconds per claim | 22.3 | 22.8 |
| The whole run | 32 s | 33 s |
| Cost per claim | $0.0018 | $0.0019 |
| Accuracy | 80% | 100% |
| Straight through | 80% | 80% |

1. **Time per claim is the same on both.** About 22 seconds a claim either way. Almost all of it is the model's calls
   to OpenRouter, which take the same time from Runtime as from the laptop. Runtime adds no time a handler would notice.
2. **The whole run took 32 seconds on Runtime, against 33 on the laptop.** The five claims run side by side, so the
   run takes about as long as its slowest claim. Runtime's new session started in about a second: the agent fetched
   its credentials from AgentCore Identity and opened its pool on the first request.
3. **Cost per claim is the same, under a fifth of a cent.** The small difference is the model's own variation. Runtime
   also bills for the session's own time, which this measure leaves out.
4. **Accuracy differs by one claim.** One of five claims was decided differently on Runtime. With five claims, one
   claim is 20 points, so this is the model's run-to-run noise, not the hosting. The whole held-out set would show it
   better.
5. **The judge didn't grade the Runtime run.** The judge runs in the app, and the app on Lambda has no OpenRouter key
   on this route, so its judge cost is $0 and there's no grade. On the laptop the judge cost $0.00015.
6. **The logs show one session.** The runtime's log group had one stream for the evaluation,
   `runtime-logs-undercover-…`, named after the session id the app made. The entrypoint answered in 1.3 seconds and
   ran the evaluation's work as an async task, which kept the session busy until the work finished.